# Concrete Strength Prediction — Training on Colab

**Архитектура хранения:**
- `/content/concrete-strength/` — git-репозиторий (локально, быстро)
- `/content/drive/MyDrive/concrete_project/` — **persistence**: чекпоинты `.pt`, JSON-логи

Colab локальный диск сбрасывается при отключении, Drive — нет.

## Ячейка 1: Монтировать Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Создаём папки на Drive для persistent хранения
import os
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/experiments', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/neat_cache', exist_ok=True)

print('Drive смонтирован.')
print(f'Persistent storage: {DRIVE_DIR}')

## Ячейка 2: Клонировать репо в локальный /content

In [ ]:
import os

# !! ЗАМЕНИ НА СВОЙ USERNAME !!
GITHUB_USERNAME = 'YOUR_USERNAME'
REPO_NAME = 'concrete-strength'

# Репо клонируем ЛОКАЛЬНО в /content — здесь живёт код
# Drive — только для артефактов (веса, логи)
REPO_DIR = f'/content/{REPO_NAME}'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Репо существует, тянем последнее...')
    !cd {REPO_DIR} && git pull
else:
    print('Клонируем репо...')
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Рабочая директория: {os.getcwd()}')
!ls -la

## Ячейка 3: Установить пакет и проверить GPU

In [ ]:
import os

# Убеждаемся что мы в правильной директории
REPO_DIR = '/content/concrete-strength'
os.chdir(REPO_DIR)
print(f'CWD: {os.getcwd()}')

# Устанавливаем пакет (editable mode)
!pip install -e '.' -q

# Проверяем GPU
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory
    print(f'VRAM: {mem/1e9:.1f} GB')
else:
    print('GPU не найден — убедись, что в Runtime > Change runtime type выбрано GPU T4')

## Ячейка 4: Smoke test + настройка путей

In [ ]:
import os
REPO_DIR = '/content/concrete-strength'
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'
os.chdir(REPO_DIR)

# Smoke test — убеждаемся что пайплайн работает
!python run_experiment.py --mode smoke_test

print('\nSmoke test прошёл! Можно запускать эксперименты.')

## Ячейка 5: Supervised Grid Search (~30-50 мин на T4)

In [ ]:
import os
REPO_DIR = '/content/concrete-strength'
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'
os.chdir(REPO_DIR)

# Результаты сохраняем сразу на Drive (persistence)
!python run_experiment.py \
    --mode supervised_grid \
    --output_dir {DRIVE_DIR}/experiments

## Ячейка 6: GAN Tuning (~75 мин на T4)

In [ ]:
import os
REPO_DIR = '/content/concrete-strength'
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'
os.chdir(REPO_DIR)

!python run_experiment.py \
    --mode gan_tune \
    --top_k 3 \
    --output_dir {DRIVE_DIR}/experiments

## Ячейка 7: Full pipeline (лучший конфиг)

In [ ]:
import os
REPO_DIR = '/content/concrete-strength'
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'
os.chdir(REPO_DIR)

!python run_experiment.py \
    --mode full_pipeline \
    --epochs 1000 \
    --output_dir {DRIVE_DIR}/experiments

## Ячейка 8: Просмотр результатов

In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'

from materialgen.tracker import ExperimentTracker
tracker = ExperimentTracker(f'{DRIVE_DIR}/experiments')
print(tracker.summary_table())

best = tracker.best_run(metric='mae')
if best:
    print(f'\nЛучший прогон: {best["experiment"]}')
    print(f'  MAE = {best["metrics"]["mae"]:.2f} MPa')
    print(f'  R2  = {best["metrics"]["r2"]:.4f}')
    print(f'  Config: {best["config"]}')

## Ячейка 9: Закоммитить результаты в Git

In [ ]:
import os
REPO_DIR = '/content/concrete-strength'
DRIVE_DIR = '/content/drive/MyDrive/concrete_project'
os.chdir(REPO_DIR)

# Копируем JSON-логи из Drive в репо для коммита
import shutil, glob
os.makedirs('experiments', exist_ok=True)
for f in glob.glob(f'{DRIVE_DIR}/experiments/*.jsonl'):
    shutil.copy2(f, 'experiments/')
for f in glob.glob(f'{DRIVE_DIR}/experiments/checkpoints/*.json'):
    os.makedirs('experiments/checkpoints', exist_ok=True)
    shutil.copy2(f, 'experiments/checkpoints/')

# Коммит (только JSON, не .pt)
!git add experiments/
!git commit -m 'Add experiment results from Colab [skip ci]' || echo 'Nothing to commit'
!git push